In [ ]:
%matplotlib module://ipympl.backend_nbagg
import SimpleITK as sitk
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np


import sys
sys.path.insert(0, "/workspace/segmentation")
from tools import img_show3D, window, img_show2D, img_show3D_interactive


In [ ]:
img_dir = Path ("/workspace/segmentation/data/practice/image_quadra/QUADRA_HC_021/retest_CT-AC.nii.gz")
img = sitk.ReadImage(img_dir)

mask_dir = Path ("/workspace/segmentation/data/practice/mask_quadra/QUADRA_HC_021/retest/liver.nii.gz")
mask = sitk.ReadImage(mask_dir)

In [ ]:


sigma = 1

smooting_filter = sitk.SmoothingRecursiveGaussianImageFilter()
smooting_filter.SetSigma(sigma)

smooth_img = smooting_filter.Execute(img)



gradient_filter = sitk.GradientImageFilter()
gradient_filter.SetUseImageSpacing(True)
gradient_filter.SetUseImageDirection(False)

gradient_img = gradient_filter.Execute(smooth_img)

gradient_x = sitk.VectorIndexSelectionCast(gradient_img, 0)
gradient_y = sitk.VectorIndexSelectionCast(gradient_img, 1)
gradient_z = sitk.VectorIndexSelectionCast(gradient_img, 2)

sigma = 1

gaussian_harris = sitk.SmoothingRecursiveGaussianImageFilter()
gaussian_harris.SetSigma(sigma)


A = gaussian_harris.Execute(gradient_x * gradient_x)
B = gaussian_harris.Execute(gradient_x * gradient_y)
C = gaussian_harris.Execute(gradient_x * gradient_z)
D = gaussian_harris.Execute(gradient_y * gradient_y)
E = gaussian_harris.Execute(gradient_y * gradient_z)
F = gaussian_harris.Execute(gradient_z * gradient_z)

det_M = A*D*F + 2.0*B*C*E - A*E*E - D*C*C - F*B*B

trace_M = A + D + F

k = 0.005
harris_response = det_M - k * trace_M * trace_M * trace_M